# Track 02b (심화) — 에이전트 harness 해부

### harness란?

`ToolAgent`는 마법이 아니라, 평범한 `관찰 → 사고 → 행동` 루프 위에 **명시적인 결정(knob)** 을 얹은 것입니다. 이런 결정 묶음을 harness라고 부릅니다. Track 02a 에서 손코딩 루프와 `ToolAgent` 를 비교했다면, 여기서는 그 `ToolAgent` 를 **분해해서 각 knob을 하나씩 껐다 켜며 무엇이 달라지는지** 를 봅니다.

harness의 **5가지 결정**: **preflight(planner)** · **routing(thinking)** · **safety(턴·도구 예산)** · **finalize** · **trace**. 이 가운데 앞의 3개를 켜고 끄며 실험하고, finalize·trace 는 항상 켜 둡니다.

### 이 노트북에서 보여줄 것 — 4 harness × 3 입력 = 12 runs

같은 입력을 knob 조합이 다른 4개 harness 로 돌립니다. **먼저 한 run 을 해부(Session 3)** 해 흐름을 읽고, 이어서 **12 run 을 표·그림으로 비교(Session 4)** 해 어떤 결정을 끄면 결과가 어떻게 달라지는지 봅니다.

| 무엇을 | 어떻게 | 무엇을 확인하나 |
|---|---|---|
| **harness 4종** | `full`(전부 on) · `no_router` · `no_planner` · `bare`(전부 off + 느슨한 안전) | knob 별 효과를 분리해서 확인 |
| **입력 3종** | `single`(환산 1회) · `many_tools`(환산 5회) · `no_tool`(잡담) | knob 차이가 실제로 드러나는 상황 만들기 |
| **trace 12건** | 각 실행의 `AgentEvent` 를 모아 turn · 도구 · LLM 호출 · 예산 차단 · 지연을 집계 | **Session 3** 한 run 해부(흐름) → **Session 4** 표·그림 비교 |

핵심 장면은 **`many_tools`(5통화 환산)** 입니다. strict 안전 예산(`max_tool_invocations=3`)을 쓰는 harness 는 4·5번째 도구 호출이 막혀(`BLOCKED (budget)`) 합계를 끝내지 못하고, 느슨한 `bare` 만 완주합니다 → **안전 예산이 작업 완성도를 직접 좌우한다**는 것을 확인합니다.

### 이 노트북을 마치면

- 에이전트의 동작을 `AgentEvent` **trace** 로 읽을 수 있습니다 (단계 / turn / 도구 / LLM 호출 / 예산 차단).
- **증상 → 어떤 knob 때문인지** 짚을 수 있습니다 (예: 큰 작업이 중간에 끊기면 → 안전 예산 / LLM 호출이 많으면 → planner·router).
- 안전 예산을 **작업 규모에 맞춰** 잡는 기준을 익힙니다 (`_out/checklist.md` 운영 체크리스트).

> **한 줄 요약:** `ToolAgent`의 기본값을 그냥 쓰지 말고, **각 knob이 맡는 역할을 이해하기.**

- **구성:** 각 `Session`은 가이드(텍스트) → 코드 → 출력 해석(텍스트) 순입니다.
- **필요:** EXAONE API 키 (12 runs · planner/finalize 때문에 **실제 LLM 호출은 12회보다 많음** — Session 4 의 `l` 열, 약 3~5분 소요).
- **산출물:** `_out/anatomy_matrix.json`, `_out/sequence_*.md`, `_out/checklist.md`


In [ ]:
import json
import os
import sys
import time
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

import logging
import warnings

# (en) Quiet library logs for readable notebook output.
# (kr) 노트북 출력을 읽기 쉽게 라이브러리 로그를 줄인다.
for _log_name in ("exaone", "exaone.llm", "exaone.llm.exaone_client", "urllib3"):
    logging.getLogger(_log_name).setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

# (en) Requires editable install at repo root: pip install -r requirements.txt && pip install -e ./exaone
# (kr) 저장소 루트에서 editable 설치 필요: pip install -r requirements.txt && pip install -e ./exaone
try:
    import exaone
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "exaone 가 설치되지 않았습니다. 저장소 루트에서 "
        "pip install -r requirements.txt && pip install -e ./exaone 후 커널을 재시작하세요."
    ) from exc

exaone.load_project_env()
# (en) Gate live LLM steps on API key presence.
# (kr) API 키 유무로 라이브 LLM 단계를 가드한다.
HAS_API = bool(os.environ.get("EXAONE_API_KEY", "").strip())
ROOT = exaone.project_root()
TRACK02 = ROOT / "recipes" / "track02_minimum_agent_loop"
DATA = TRACK02 / "data"

base_url = os.environ.get("EXAONE_BASE_URL", "").strip() or "http://localhost:8000/v1"
model = (
    os.environ.get("EXAONE_MODEL", "").strip() or exaone.llm.ExaoneClient.DEFAULT_MODEL
)
client = exaone.integrations.build_llm_from_env()
print("model:", client.model)

_RATES = json.loads((DATA / "exchange_rates.json").read_text(encoding="utf-8"))
EXCHANGE_RATES_KRW = _RATES["rates"]
AS_OF = _RATES["as_of"]


def _rate(base: str, quote: str) -> float:
    base, quote = base.upper(), quote.upper()
    if base == "KRW" and quote == "KRW":
        return 1.0
    if base == "KRW":
        return 1.0 / EXCHANGE_RATES_KRW[quote]
    if quote == "KRW":
        return EXCHANGE_RATES_KRW[base]
    return EXCHANGE_RATES_KRW[base] / EXCHANGE_RATES_KRW[quote]


def _exec_rate(args: dict) -> dict:
    base, quote = (args.get("base") or "").upper(), (args.get("quote") or "KRW").upper()
    table = set(EXCHANGE_RATES_KRW) | {"KRW"}
    if base not in table or quote not in table:
        return {"error": f"unknown currency: base={base}, quote={quote}"}
    return {
        "base": base,
        "quote": quote,
        "rate": round(_rate(base, quote), 4),
        "as_of": AS_OF,
    }


def _exec_convert(args: dict) -> dict:
    amount, base, quote = (
        args.get("amount"),
        (args.get("base") or "").upper(),
        (args.get("quote") or "KRW").upper(),
    )
    table = set(EXCHANGE_RATES_KRW) | {"KRW"}
    if not isinstance(amount, (int, float)):
        return {"error": "amount must be a number"}
    if base not in table or quote not in table:
        return {"error": "unknown currency"}
    return {
        "amount": amount,
        "base": base,
        "quote": quote,
        "converted": round(float(amount) * _rate(base, quote), 2),
        "rate_used": round(_rate(base, quote), 4),
    }


def build_registry() -> "exaone.tools.ToolRegistry":
    reg = exaone.tools.ToolRegistry()
    reg.register(
        exaone.tools.Tool(
            name="exchange_rate",
            execute=_exec_rate,
            schema={
                "type": "function",
                "function": {
                    "name": "exchange_rate",
                    "description": "Get exchange rate base to quote.",
                    "parameters": {
                        "type": "object",
                        "required": ["base"],
                        "properties": {
                            "base": {"type": "string"},
                            "quote": {"type": "string", "default": "KRW"},
                        },
                        "additionalProperties": False,
                    },
                },
            },
        )
    )
    reg.register(
        exaone.tools.Tool(
            name="convert_money",
            execute=_exec_convert,
            schema={
                "type": "function",
                "function": {
                    "name": "convert_money",
                    "description": "Convert amount of base into quote.",
                    "parameters": {
                        "type": "object",
                        "required": ["amount", "base"],
                        "properties": {
                            "amount": {"type": "number"},
                            "base": {"type": "string"},
                            "quote": {"type": "string", "default": "KRW"},
                        },
                        "additionalProperties": False,
                    },
                },
            },
        )
    )
    return reg


SYSTEM_PROMPT = (
    "You are a Korean-speaking currency assistant. "
    "Use exchange_rate / convert_money tools when amounts or rates are mentioned. Reply in Korean."
)

out_dir = Path("_out")
out_dir.mkdir(parents=True, exist_ok=True)
print("준비 완료. HAS_API =", HAS_API)

**출력 해석:** `model: …` 과 `준비 완료. HAS_API = True` 두 줄이 보이면 이 노트북에서 쓸 client·DATA·`_out` 경로가 모두 준비된 것입니다. (모델 이름이 `.env`의 `EXAONE_MODEL`과 일치하는지 확인하세요.)


## Session 1. harness 변형 4종

**세 가지 토글**(preflight · routing · safety_strict)을 바꿔 **같은 입력** 에서 결과가 어떻게 달라지는지 봅니다. (나머지 두 결정인 finalize·trace 는 항상 켜 두므로, 노트북 제목의 "5가지 결정" 중 그리드에서 실제로 끄고 켜는 것은 이 세 가지입니다.)

| 이름 | preflight | routing | safety_strict | 의도 |
|---|---|---|---|---|
| `full`       | on  | on  | on  | 운영 권장. 모든 결정을 활성화. |
| `no_router`  | on  | off | on  | 라우터 효과 격리. thinking 결정은 모델 기본값에 맡김. |
| `no_planner` | off | on  | on  | preflight·next-step hint 없이 enrich 루프만 사용. |
| `bare`       | off | off | off | 손코딩 루프와 가장 가까운 형태(all off + loose safety). |

세 입력 (`single` / `many_tools` / `no_tool`) × 4 harness = **12 runs**. 입력은 knob 별 차이가 표에서 드러나도록 골랐습니다: `single`은 기준선, `many_tools`는 환산 5회가 필요해 **strict 안전 예산(max_tool 3)** 을 넘겨 safety 효과를 드러내고, `no_tool`은 도구가 필요 없는 잡담입니다.


### Session 1-1. HarnessConfig·입력 정의

**하는 일:** harness 4종(토글 3개)과 입력 3종을 정의합니다.

**정상:** `4 harnesses × 3 inputs = 12 runs` 가 보임

**의미:** 무엇을 끄면 결과가 어떻게 달라지는지 비교할 실험 구성을 만듭니다.


In [ ]:
# (en) One place to flip the three harness toggles; safety_strict tightens turn/tool budgets.
# (kr) 세 토글을 한 곳에서 켜고 끈다. safety_strict 는 턴·도구 예산을 빡빡하게 조인다.
@dataclass
class HarnessConfig:
    name: str
    preflight: bool = True
    routing: bool = True
    safety_strict: bool = True
    description: str = ""


def make_agent(cfg: HarnessConfig) -> "exaone.agents.ToolAgent":
    return exaone.agents.ToolAgent(
        tool_registry=build_registry(),
        system_prompt=SYSTEM_PROMPT,
        max_turns=2 if cfg.safety_strict else 6,
        max_tool_invocations=3 if cfg.safety_strict else 12,
        use_thinking_router=cfg.routing,
        use_next_step_planner=cfg.preflight,
    )


HARNESSES = [
    HarnessConfig(
        "full", preflight=True, routing=True, safety_strict=True, description="all on"
    ),
    HarnessConfig(
        "no_router",
        preflight=True,
        routing=False,
        safety_strict=True,
        description="router off",
    ),
    HarnessConfig(
        "no_planner",
        preflight=False,
        routing=True,
        safety_strict=True,
        description="planner off",
    ),
    HarnessConfig(
        "bare",
        preflight=False,
        routing=False,
        safety_strict=False,
        description="all off + loose safety",
    ),
]

# (en) Three inputs chosen so each knob can actually move a column:
#      single  = baseline (all harnesses agree),
#      many_tools = 5 conversions > strict budget (max_tool_invocations=3) -> safety bites,
#      no_tool = chit-chat (planner suppresses tools).
# (kr) knob 별로 표의 한 열을 실제로 움직이도록 고른 세 입력:
#      single = 기준선(전 harness 동일),
#      many_tools = 환산 5회 > strict 예산(max_tool_invocations=3) -> 안전 예산이 물림,
#      no_tool = 잡담(planner 가 도구를 억제).
INPUTS = [
    {"id": "single", "query": "100달러는 원화로 얼마야?"},
    {
        "id": "many_tools",
        "query": (
            "달러 100, 유로 50, 엔 10000, 위안 200, 파운드 20을 "
            "각각 원화로 환산한 다음 전부 합쳐서 원화로 얼마인지 알려줘."
        ),
    },
    {"id": "no_tool", "query": "환율이 왜 매일 변동하는지 한 문장으로 설명해줘."},
]

print(
    f"{len(HARNESSES)} harnesses × {len(INPUTS)} inputs = {len(HARNESSES) * len(INPUTS)} runs"
)

**출력 해석:** `4 harnesses × 3 inputs = 12 runs`가 보이면 이 단계는 통과입니다.



## Session 2. 그리드 실행

각 (harness, 입력) 조합을 `run_stream` 으로 돌리고, 흘러나온 `AgentEvent` 를 모아 둡니다. 이벤트를 세면 turn 수·도구 호출 수·LLM 호출 수를 계산할 수 있습니다.

> **참고 — `Tool invocation budget exhausted` 로그는 정상입니다(에러 아님).** `many_tools` 를 strict 안전 예산 구성(`full`·`no_router`·`no_planner`)으로 돌리면 환산 5번 중 4·5번째가 `max_tool_invocations=3` 에 막혀 `Tool execution failed: … — Tool invocation budget exhausted.` 가 한 줄씩(총 2줄) 찍힙니다. 이는 **안전 예산이 작동하는 모습** 이며 바로 이 노트북이 보여주려는 장면입니다(Session 4 의 `b` 열·`F` 의 근거). `bare`(예산 12)에서는 출력되지 않고, run 자체는 정상 종료합니다(`harness_error`=None).


### Session 2-1. 12 runs 수집

**하는 일:** 각 (harness, 입력) 조합을 `run_stream` 으로 돌립니다.

**정상:** 각 줄에 `turns=` · `tools=` · `lat=` 가 보임

**의미:** 이벤트를 모으면 turn·도구·지연이 숫자로 나옵니다.


In [ ]:
# (en) Collect every AgentEvent for one (harness, input) run; we count event types later.
# (kr) 한 (harness, 입력) 실행의 모든 AgentEvent 를 모은다. 이벤트 타입은 뒤에서 센다.
@dataclass
class RunAnatomy:
    harness: str
    input_id: str
    events: list = field(default_factory=list)
    final_answer: str = ""
    error: str = None
    latency_ms: float = 0.0
    llm_calls: int = 0


def collect(cfg: HarnessConfig, query: str) -> RunAnatomy:
    agent = make_agent(cfg)
    rec = RunAnatomy(harness=cfg.name, input_id="")
    t0 = time.monotonic()
    try:
        for ev in agent.run_stream(
            exaone.agents.AgentContext(query=query),
            llm=client,
            stream_llm=False,
            stream_enrich_reasoning=False,
        ):
            rec.events.append(ev)
            if ev.type == "run_end":
                rec.final_answer = ev.payload.get("final_content", "") or ""
                # (en) True LLM-call count incl. planner/router phases (llm_end events miss those).
                # (kr) planner/router 단계까지 포함한 실제 LLM 호출 수(llm_end 이벤트는 그걸 놓침).
                rec.llm_calls = len(ev.payload.get("llm_calls") or [])
                if ev.payload.get("error"):
                    rec.error = ev.payload.get("error")
    except Exception as e:
        rec.error = f"{type(e).__name__}: {e}"
    rec.latency_ms = (time.monotonic() - t0) * 1000
    return rec


grid = {}
for cfg in HARNESSES:
    for inp in INPUTS:
        print(f"running [{cfg.name:<11}] {inp['id']:<10} ", end="", flush=True)
        rec = collect(cfg, inp["query"])
        rec.input_id = inp["id"]
        grid[(cfg.name, inp["id"])] = rec
        tools = sum(1 for ev in rec.events if ev.type == "tool_start")
        turns = sum(1 for ev in rec.events if ev.type == "turn_start")
        err = f"  ERR={rec.error}" if rec.error else ""
        print(
            f"turns={turns} tools={tools} llm={rec.llm_calls} lat={rec.latency_ms:.0f}ms{err}"
        )
print(f"\ndone. {len(grid)} runs.")

**출력 해석:** 각 줄에 `turns`·`tools`·`llm`·`lat`이 보이면 이 단계는 통과입니다. 단 `tools`는 *시도* 횟수이므로 예산에 막힌 호출도 포함합니다(실제 예산 차단은 Session 4 의 `b` 열). 합격/불합격과 knob별 비교는 Session 4 에서 합니다. (`llm`은 planner·router 까지 포함한 실제 LLM 호출 수입니다.)


## Session 3. 한 실행 해부 (trace 읽기)

Session 4 표가 여러 run 을 *개수* 로 비교한다면, 여기서는 **한 run** 을 골라 그 **내부 흐름(trace)** 을 시퀀스 다이어그램으로 해부합니다. harness 가 질의 하나를 처리하며 거치는 **단계(phase: preflight → enrich → finalize)** · **turn** · 각 단계의 **이벤트**(LLM 호출 · 도구 호출 · 결과)를 한눈에 보고, **어느 단계에서 어떤 결정(knob)이 작동하는지** 를 짚습니다. 이 trace 는 운영 관측성(observability)의 기본 단위입니다 (→ Track 07).


### Session 3-1. trace 다이어그램

**하는 일:** `full`의 `single` 실행 한 건을 `display(Markdown)`으로 그립니다.

**정상:** 시퀀스 다이어그램 1개 출력 (Mermaid 지원 뷰어면 그림, 아니면 mermaid 소스)

**의미:** 한 run이 preflight → enrich → finalize 로 흐르는 **골격** 과 단계별 이벤트(LLM·도구 호출)를 확인합니다.


In [ ]:
from IPython.display import Markdown, display


# (en) One-line, Mermaid-safe label: drop newlines / angle brackets / quotes / braces that
#      would otherwise break sequenceDiagram parsing, then truncate.
# (kr) 한 줄·Mermaid 안전 라벨: sequenceDiagram 파싱을 깨뜨리는 줄바꿈·꺽쇠·따옴표·중괄호를
#      제거한 뒤 잘라낸다.
def _mm(s, n: int = 48) -> str:
    s = str(s).replace("\n", " ").replace("\r", " ")
    for ch in '<>"{}':
        s = s.replace(ch, "")
    s = " ".join(s.split())
    return (s[:n] + "…") if len(s) > n else s


# (en) Summarize a tool result so blocked vs ok is obvious in the diagram.
# (kr) 도구 결과를 요약해 차단/정상이 다이어그램에서 바로 보이게 한다.
def _tool_outcome(preview: str) -> str:
    low = (preview or "").lower()
    if "budget" in low or "exhausted" in low:
        return "BLOCKED (budget)"
    if '"error"' in (preview or ""):
        return "error"
    return "ok"


# (en) Turn one run's event stream into a Mermaid sequence diagram of harness <-> LLM <-> tools.
#      Phases / turns / events show the run's anatomy; budget-blocked calls (used by the Session 4
#      picture) are wrapped in a translucent red rect (rgba keeps text readable on dark + light).
# (kr) 한 실행의 이벤트 흐름을 harness <-> LLM <-> tools 시퀀스 다이어그램으로 바꾼다.
#      phase/turn/event 가 run 의 해부도이며, 예산 차단 호출(Session 4 그림에서 사용)은 반투명
#      빨간 rect 로 감싼다(rgba 라 다크·라이트 양쪽서 글씨가 읽힌다).
def to_mermaid(rec: RunAnatomy) -> str:
    n_blocked = sum(
        1
        for ev in rec.events
        if ev.type == "tool_end"
        and _tool_outcome(ev.payload.get("result_preview") if ev.payload else "")
        == "BLOCKED (budget)"
    )
    n_ok = sum(
        1
        for ev in rec.events
        if ev.type == "tool_end"
        and _tool_outcome(ev.payload.get("result_preview") if ev.payload else "")
        == "ok"
    )
    lines = [
        "sequenceDiagram",
        "    autonumber",
        "    participant U as User",
        "    participant H as Harness",
        "    participant P as Planner/Router",
        "    participant L as LLM",
        "    participant T as Tools",
        f"    U->>H: query [{rec.input_id}] ({rec.harness})",
        f"    Note over U,T: {n_ok} tool(s) done, {n_blocked} blocked by safety budget",
    ]
    cur_phase = None
    for ev in rec.events:
        t, p = ev.type, ev.payload or {}
        if t == "phase_start":
            ph = p.get("phase")
            if ph != cur_phase:
                lines.append(f"    Note over H: phase = {ph}")
                cur_phase = ph
        elif t == "planner_end":
            # (en) planner_end has two kinds: catalog_screen (answerable) vs evaluate_progress (action/finalize).
            # (kr) planner_end 는 두 종류다: catalog_screen(answerable) 와 evaluate_progress(action/finalize).
            if p.get("kind") == "catalog_screen":
                lines.append(
                    f"    H->>P: catalog_screen -> answerable={p.get('answerable')}"
                )
            else:
                lines.append(
                    f"    H->>P: evaluate_progress -> action={p.get('action')}, finalize={p.get('should_finalize')}"
                )
        elif t == "turn_start":
            lines.append(f"    Note over H: turn {ev.turn}")
        elif t == "llm_end":
            tag = "with tool_calls" if p.get("has_tool_calls") else "final-ish"
            lines.append(
                f"    H->>L: chat ({tag}, latency={round(p.get('latency_ms') or 0)}ms)"
            )
        elif t == "tool_start":
            lines.append(f"    H->>T: {p.get('tool')}({_mm(p.get('args_preview'))})")
        elif t == "tool_end":
            outcome = _tool_outcome(p.get("result_preview"))
            if outcome.startswith("BLOCKED"):
                # (en) Wrap this blocked call (its tool_start + result) in a translucent red box.
                # (kr) 막힌 호출(직전 tool_start + 결과)을 반투명 빨간 박스로 감싼다.
                started = lines.pop() if lines and "->>T:" in lines[-1] else None
                lines.append("    rect rgba(255, 0, 0, 0.3)")
                if started:
                    lines.append(started)
                lines.append(f"    T-->>H: {outcome}")
                lines.append("    end")
            else:
                lines.append(f"    T-->>H: {outcome}")
        elif t == "error":
            # (en) Standalone error event (e.g. safety budget exhausted) — show it in the diagram too.
            # (kr) 독립 error 이벤트(예: 안전 예산 소진)도 다이어그램에 표시한다.
            lines.append(f"    Note over H: ERROR {_mm(p.get('message'))}")
        elif t == "run_end":
            ans = _mm(p.get("final_content"), 60)
            if p.get("error"):
                lines.append(f"    Note over H: error = {_mm(p.get('error'))}")
            lines.append(f"    H-->>U: answer = {ans}")
    return "\n".join(lines)


# (en) Dissect ONE clean run: full harness on the simple `single` query — its phase flow
#      (preflight -> enrich -> finalize), turns, and per-step events, with no budget noise.
#      (The full-vs-bare budget picture lives next to the Session 4 table.)
# (kr) 깨끗한 한 실행을 해부한다: full harness 의 단순 `single` 질의 — preflight -> enrich
#      -> finalize 단계 흐름·turn·단계별 이벤트가 예산 잡음 없이 보인다.
#      (full↔bare 예산 그림은 Session 4 표 옆에 둔다.)
rec = grid[("full", "single")]
display(
    Markdown(
        f"**=== full / single (trace 해부) ===**\n\n```mermaid\n{to_mermaid(rec)}\n```"
    )
)

**출력 해석:** 위→아래로 한 run의 전체 흐름이고, **단계마다 어떤 결정이 작동하는지** 가 보입니다:

- **preflight** — planner가 '질의가 도구로 답할 수 있는지' 를 선별 (planner 결정)
- **enrich** — reason↔tool 루프: LLM 이 `chat (with tool_calls)` 로 도구를 부르고(`convert_money`) 결과를 다시 읽음 (router/thinking·safety 가 작동하는 구간)
- **finalize** — reasoning 채널을 제거하고 사용자 답 생성 (finalize 결정)

즉 harness의 결정들이 **타임라인 어디에서** 작동하는지가 한 그림에 들어옵니다 — 이것이 운영 trace 를 읽는 기본 단위입니다 (→ Track 07).


## Session 4. 그리드 표

세로축은 입력, 가로축은 harness 입니다. 셀은 `t{turns}/k{tools}/b{blocked}/l{llm_calls} {P|F}` 입니다 (`b` = 안전 예산에 막힌 도구 호출 수, `l` = planner·router 까지 포함한 **실제** LLM 호출 수).

knob 별로 표의 *어느 열* 을 봐야 차이가 보이는지 정리합니다(효과가 또렷한 순서):

1. **safety_strict (`bare` 만 off) — `b`·`P/F` 에서 가장 또렷** — `many_tools`는 환산 5번이 필요한데, strict 3종(`full`·`no_router`·`no_planner`)은 `max_tool_invocations=3`이라 4번째부터 **예산에 막혀**(`b>0`) 합계를 끝내지 못해 **F**, 느슨한 `bare`(=12)만 완주해 **P** 입니다. → 안전 예산은 무조건 낮출 수 있는 값이 아니라 **작업 규모에 맞춰 정해야 하는 값**입니다.
2. **routing (`no_router` 만 off) — `l` 에서 또렷** — thinking router는 라우팅 결정에 LLM 을 한 번 더 씁니다. `full` vs `no_router` 를 보면 **모든 입력에서 `l`이 정확히 +1**(single 4↔3, no_tool 2↔1, many_tools 3↔2)입니다. 즉 router는 쉬운 입력에서도 매 실행 LLM 호출 1회의 비용이 있습니다.
3. **preflight (`no_planner` 만 off) — 약함** — planner가 켜져도 router 와 함께면 catalog 선별이 **통합(unified)** 되어 `l` 추가 호출이 거의 안 보입니다(`full`≈`no_planner`). 효과는 `turns` 에서만 약하게 드러납니다. planner의 finalize 게이트 때문에 `full`이 `no_planner`보다 turn 이 적기도 합니다(single t2 vs t3; 샘플링에 따라 ±1).


### Session 4-1. P/F 요약 표

**하는 일:** 12 runs 를 `t{turns}/k{tools}/b{blocked}/l{llm_calls}` 표로 요약합니다.

**정상:** 입력별 harness 열이 `P` 또는 `F`

**의미:** 한눈에 어떤 토글이 성공·비용에 영향을 주는지 봅니다 (특히 `many_tools` 행의 `b`·`P/F`).


In [ ]:
# (en) Pass rules per input: single needs the right number; many_tools must finish every
#      conversion within the tool budget (no "budget exhausted"); no_tool must use no tool.
# (kr) 입력별 통과 기준: single 은 정답 수치, many_tools 는 예산 안에서 모든 환산 완주
#      (예산 소진 없음), no_tool 은 도구 미사용.
EXPECTED_TOKENS = {
    "single": ["138,050", "138050"],
    "no_tool": [],
}


def _budget_blocked(rec: "RunAnatomy") -> int:
    # (en) Count tool results rejected by the invocation budget (the safety cap biting).
    # (kr) 호출 예산(안전 상한)에 막힌 도구 결과 수를 센다.
    return sum(
        1
        for ev in rec.events
        if ev.type == "tool_end"
        and any(
            kw in (ev.payload.get("result_preview") or "").lower()
            for kw in ("budget", "exhausted")
        )
    )


def summarize(rec: RunAnatomy) -> dict:
    types = Counter(ev.type for ev in rec.events)
    tools_used = types["tool_start"]
    answer = rec.final_answer or ""
    blocked = _budget_blocked(rec)
    if rec.input_id == "many_tools":
        # (en) Pass only when the harness let every conversion run (loose safety budget).
        # (kr) 모든 환산을 예산 안에서 끝냈을 때만 통과(느슨한 안전 예산).
        passed = blocked == 0 and tools_used >= 4 and rec.error is None
    elif EXPECTED_TOKENS.get(rec.input_id):
        passed = any(tok in answer for tok in EXPECTED_TOKENS[rec.input_id])
    else:
        passed = tools_used == 0 and len(answer) > 5 and rec.error is None
    return {
        "turns": types["turn_start"],
        "tools": tools_used,
        "blocked": blocked,
        # (en) True LLM calls (planner+router+enrich+finalize); falls back to llm_end if unavailable.
        # (kr) 실제 LLM 호출 수(planner+router+enrich+finalize); 없으면 llm_end 로 대체.
        "llm_calls": rec.llm_calls or types["llm_end"],
        "phases": types["phase_start"],
        "errors": types["error"],
        "latency_ms": round(rec.latency_ms, 1),
        "pass": passed,
        "final_answer": answer,
        "harness_error": rec.error,
    }


matrix = {inp["id"]: {} for inp in INPUTS}
print(f"{'input':<11} " + " ".join(f"{h.name:<22}" for h in HARNESSES))
print("-" * (12 + 23 * len(HARNESSES)))
for inp in INPUTS:
    cells_out = []
    for cfg in HARNESSES:
        s = summarize(grid[(cfg.name, inp["id"])])
        matrix[inp["id"]][cfg.name] = s
        cells_out.append(
            f"t{s['turns']}/k{s['tools']}/b{s['blocked']}/l{s['llm_calls']} {'P' if s['pass'] else 'F'}"
        )
    print(f"{inp['id']:<11} " + " ".join(f"{c:<22}" for c in cells_out))

**출력 해석:** `single`·`no_tool`은 네 harness 모두 `P` 입니다(정확성은 knob 과 무관). knob 효과는 열별로 봅니다:

- **safety** — `many_tools` 에서 strict 3종은 예산 차단(`b>0`, 이번 실행 `b2`)으로 `F`, `bare` 만 `b0`/`P` (pass_rate 0.667 vs 1.0). 핵심 교훈: **안전 예산이 완성도를 좌우**.
- **routing** — `l` 열에서 `full`이 `no_router`보다 매 입력 +1 (라우팅에 LLM 1회 더).
- **preflight** — router 와 통합되어 `l` 에는 거의 보이지 않고, `turns` 에서만 약하게 드러남.

(`full`의 낮은 pass_rate 는 결함이 아니라 이 작업에 비해 예산이 작게 잡힌 것입니다. 운영에서는 예상 도구 호출 수의 약 2배로 잡는 편이 안전합니다.)


### Session 4-2. 예산 차단 시각화 (표 ↔ 그림)

**하는 일:** 위 표에서 결과가 갈린 `many_tools`의 `full`(strict) vs `bare`(loose)를 시퀀스 다이어그램으로 그립니다.

**정상:** 다이어그램 2개 — `full`은 빨간 박스(예산 차단, 보통 2개), `bare`는 0개.

**의미:** 표의 `b>0/F` ↔ `b0/P` 가 그림에선 *빨간 박스 유무* 로 보입니다. 표는 **몇 개**가 막혔는지를, 그림은 흐름의 **어디서** 막히는지를 보여줍니다.


In [ ]:
# (en) Pair the table with the picture: full (strict) blocks the late calls, bare (loose) runs all 5.
# (kr) 표와 그림을 짝짓는다: full(strict)은 뒤쪽 호출이 예산에 막히고, bare(loose)는 5개를 완주.
for hname in ("full", "bare"):
    rec = grid[(hname, "many_tools")]
    display(
        Markdown(
            f"**=== {hname} / many_tools ===**\n\n```mermaid\n{to_mermaid(rec)}\n```"
        )
    )

**출력 해석:** 표(개수) ↔ 그림(흐름) 대응:

- **빨간 박스 = 안전 예산에 막혀 실행되지 않은 도구 호출** (`max_tool_invocations` 초과 → `BLOCKED (budget)`)
- **`full`** (예산 3): 빨간 박스로 2개 차단 → 표의 `b>0/F` 와 일치, 합계 불완전
- **`bare`** (예산 12): 박스 0개 → 표의 `b0/P` 와 일치, 합계 완성(372,480원)
- (Mermaid 미지원 뷰어면 `rect rgba(...)` 소스가 보이고, `_out/sequence_*.md`는 GitHub 에서 렌더링)


## Session 5. 산출물 저장


### Session 5-1. JSON·MD·checklist

**하는 일:** `anatomy_matrix.json` · `sequence_*.md`(입력 3종) · `checklist.md` 를 저장합니다.

**정상:** `saved:` **다섯 줄**(matrix 1 + sequence 3 + checklist 1) + `[summary]` JSON

**의미:** Track 07/08 회귀·비교에 쓸 trace 산출물입니다.


In [ ]:
matrix_payload = {
    "model": client.model,
    "harnesses": [
        {
            "name": h.name,
            "preflight": h.preflight,
            "routing": h.routing,
            "safety_strict": h.safety_strict,
            "description": h.description,
        }
        for h in HARNESSES
    ],
    "inputs": INPUTS,
    "grid": matrix,
    "summary": {
        "pass_rate_by_harness": {
            h.name: round(
                sum(1 for inp in INPUTS if matrix[inp["id"]][h.name]["pass"])
                / len(INPUTS),
                3,
            )
            for h in HARNESSES
        },
        "avg_latency_by_harness_ms": {
            h.name: round(
                sum(matrix[inp["id"]][h.name]["latency_ms"] for inp in INPUTS)
                / len(INPUTS),
                1,
            )
            for h in HARNESSES
        },
    },
}
mx_path = out_dir / "anatomy_matrix.json"
mx_path.write_text(
    json.dumps(matrix_payload, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("saved:", mx_path.resolve())

for inp in INPUTS:
    md_lines = [f"# Sequence — {inp['id']}: {inp['query']}", ""]
    for cfg in HARNESSES:
        rec = grid[(cfg.name, inp["id"])]
        md_lines += [
            f"## harness = `{cfg.name}` ({cfg.description})",
            "",
            "```mermaid",
            to_mermaid(rec),
            "```",
            "",
        ]
    p = out_dir / f"sequence_{inp['id']}.md"
    p.write_text("\n".join(md_lines), encoding="utf-8")
    print("saved:", p.resolve())

checklist = """# Track 02 — Operating Checklist

5-decision rubric to consciously turn the knobs of an EXAONE ToolAgent harness.

## Before deploying
- [ ] Preflight (planner) — turn on if you expect chit-chat / non-tool inputs mixed in.
- [ ] Routing (thinking router) — turn on if inputs have varying difficulty (costs ~1 extra LLM call per run).
- [ ] Safety
  - [ ] max_turns <= 4 unless you have a measured reason for more.
  - [ ] max_tool_invocations <= 2x expected tool calls per query (too tight cuts off multi-tool tasks -> blocked).
  - [ ] Tool functions must never raise on bad input; return {"error": ...} instead.
- [ ] Finalize — leave on (default). Strip reasoning channel before returning to user.
- [ ] Trace — keep call-trace plumbing on; needed by Track 07-08 metrics.

## When something feels off (signals this notebook exposes)
- Empty answer -> check the run's harness_error (run_end.payload.error) and the matrix `errors` count.
- Multi-tool task cut short / wrong total -> check the `b` (blocked) column or "budget exhausted" logs; raise max_tool_invocations.
- High cost -> compare the `l` (llm_calls = planner+router+enrich+finalize) column and summary.avg_latency_by_harness_ms; routing adds ~1 LLM call per run.
"""
cl_path = out_dir / "checklist.md"
cl_path.write_text(checklist, encoding="utf-8")
print("saved:", cl_path.resolve())
print("\n[summary]")
print(json.dumps(matrix_payload["summary"], ensure_ascii=False, indent=2))

**출력 해석:** `saved:` 와 `[summary]`가 보이면 이 단계는 통과입니다.


## 체크포인트

- [ ] 12 runs 모두 `harness_error` 가 None (예산 소진은 *도구* 실패로 처리되므로 run 자체는 정상 종료).
- [ ] `single`·`no_tool`은 네 harness 모두 `P` (정확성은 knob 과 무관한 기준선).
- [ ] `many_tools` 에서 strict 3종(`full`·`no_router`·`no_planner`)은 `b>0` 으로 `F`, `bare` 만 `b0` 으로 `P` (안전 예산이 완성도를 좌우).
- [ ] `anatomy_matrix.json`, `sequence_*.md`, `checklist.md` 가 모두 `_out/` 에 있음.

**다음:** Track 03 — Tools & MCP (또는 Track 02 lab 복귀)


## Wrap-up. 마무리

이 노트북에서는 `ToolAgent` harness를 preflight, routing, safety 조합으로 나누고, 4가지 harness와 3가지 입력을 교차 실행해 trace·표·시퀀스 다이어그램으로 비교했습니다.

이를 통해 safety 예산은 작업 완성도에 직접 영향을 주고, routing은 LLM 호출 비용을 늘리며, preflight는 질의 선별과 finalize 흐름에 영향을 준다는 점을 확인했습니다. 운영에서는 이 세 knob을 기본값으로만 두지 말고, 입력 규모와 비용 목표에 맞춰 조정해야 합니다.
